In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import expm, eigvals
from scipy.optimize import minimize_scalar


# ============================================================
# 1) Paramètres fixes
# ============================================================

A1_FIX = 2.94
A2_FIX = 4.17
A3_FIX = 0.34
SIGMA2_FIX = 130.446803436953

# Données hourly -> 1 heure
H_FIX = 1.0


# ============================================================
# 2) Outils linéaires
# ============================================================

def matrixA(a):
    """
    Matrice compagnon, même convention que dans yuima::MatrixA :
    dernière ligne = -a[p:1]
    pour a = [a1, a2, ..., ap]
    """
    a = np.asarray(a, dtype=float)
    p = len(a)

    A = np.zeros((p, p), dtype=float)
    if p > 1:
        A[:-1, 1:] = np.eye(p - 1)
    A[-1, :] = -a[::-1]
    return A


def check_stability(A, tol=1e-12):
    lam = eigvals(A)
    lam = np.real_if_close(lam)
    stable = np.all(np.real(lam) < -tol)
    return stable, lam


def nearest_psd(M, eps=1e-12):
    """
    Projection spectrale minimale sur les matrices symétriques PSD.
    """
    M = 0.5 * (M + M.T)
    vals, vecs = np.linalg.eigh(M)
    vals = np.maximum(vals, eps)
    return (vecs * vals) @ vecs.T


# ============================================================
# 3) V_inf façon yuima::V0inf
# ============================================================

def v0inf_yuima(a, sigma2):
    """
    Reproduction Python de la logique de yuima::V0inf pour CARMA(p, q).
    Ici p = len(a). sigma2 = variance du bruit Brownien.

    Dans le code yuima, V0inf(a,p,sigma) utilise sigma^2.
    Ici on passe directement sigma2.
    """
    a = np.asarray(a, dtype=float)
    p = len(a)

    B = np.zeros((p, p), dtype=float)
    aa = -a[::-1].copy()   # aa <- -rev(a) dans yuima

    # Traduction fidèle de la double boucle R
    for i in range(p):          # i python = 0..p-1 <-> i_R = i+1
        iR = i + 1
        for j in range(p):      # j python = 0..p-1 <-> j_R = j+1
            jR = j + 1
            k = 2 * jR - iR     # 2*j - i en indices R

            if 1 <= k <= p:
                B[i, j] = ((-1.0) ** (jR - iR)) * aa[k - 1]
            if k == p + 1:
                B[i, j] = ((-1.0) ** (jR - iR - 1))

    # R: Vdiag <- -solve(B)[,p] * 0.5 * sigma^2
    e_p = np.zeros(p)
    e_p[-1] = 1.0
    Vdiag = -np.linalg.solve(B, e_p) * 0.5 * sigma2

    V = np.diag(Vdiag)

    # Complétion hors diagonale comme dans yuima
    for i in range(p):
        iR = i + 1
        for j in range(i + 1, p):
            jR = j + 1
            if ((iR + jR) % 2) == 0:
                midR = (iR + jR) // 2
                sign = (-1.0) ** ((iR - jR) // 2)
                V[i, j] = sign * V[midR - 1, midR - 1]
                V[j, i] = V[i, j]

    return nearest_psd(V, eps=1e-14)


# ============================================================
# 4) Discrétisation
# ============================================================

def discrete_matrices_yuima_style(a, h, sigma2):
    """
    Comme dans yuima::carma.kalman :
      A     = MatrixA(a)
      F     = expm(A*h)
      V_inf = V0inf(a,p,sigma)
      Q     = V_inf - F V_inf F'
    """
    A = matrixA(a)
    F = expm(A * h)
    F = np.real_if_close(F)

    V_inf = v0inf_yuima(a, sigma2)
    Q = V_inf - F @ V_inf @ F.T
    Q = nearest_psd(Q, eps=1e-14)

    return A, F, Q, V_inf


# ============================================================
# 5) Log-vraisemblance gaussienne par innovations
# ============================================================

def kalman_loglik_b0_only(
    y,
    h,
    a1,
    a2,
    a3,
    sigma2,
    b0,
    demean=True,
    return_details=False,
    psd_eps=1e-12,
    sn_min=1e-10,
):
    """
    Vraie log-vraisemblance gaussienne conditionnelle/stationnaire
    pour le CARMA(3,1) échantillonné à pas h, avec

        a1, a2, a3, sigma2 fixés
        b = [b0, 1, 0]^T
        seul b0 estimé

    Hypothèse: bruit gaussien, pas de bruit de mesure.
    """
    y = np.asarray(y, dtype=float).ravel()
    if y.size == 0:
        raise ValueError("y vide.")

    if demean:
        y = y - np.mean(y)

    a = np.array([a1, a2, a3], dtype=float)

    A, F, Q, V_inf = discrete_matrices_yuima_style(a=a, h=h, sigma2=sigma2)

    stable, lam = check_stability(A)
    if not stable:
        out = {
            "failed": True,
            "reason": "A non stable",
            "eigvals_A": lam
        }
        return out if return_details else -np.inf

    b = np.array([b0, 1.0, 0.0], dtype=float)

    # Initialisation stationnaire
    x = np.zeros(3, dtype=float)
    P = V_inf.copy()

    loglik = 0.0

    if return_details:
        innovations = np.zeros_like(y)
        innovation_vars = np.zeros_like(y)
        x_pred_hist = np.zeros((len(y), 3))
        x_filt_hist = np.zeros((len(y), 3))

    for t, yt in enumerate(y):
        # Prediction
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q
        P_pred = nearest_psd(P_pred, eps=psd_eps)

        # Innovation
        v = yt - b @ x_pred
        S = float(b @ P_pred @ b)

        if (not np.isfinite(S)) or (S <= sn_min):
            out = {
                "failed": True,
                "reason": f"Innovation variance invalide: S={S}",
                "eigvals_A": lam,
                "A": A,
                "F": F,
                "Q": Q,
                "V_inf": V_inf,
                "last_P_pred": P_pred,
                "b": b,
                "t_fail": t,
            }
            return out if return_details else -np.inf

        # Contribution de log-vraisemblance
        term = -0.5 * (np.log(2.0 * np.pi) + np.log(S) + (v * v) / S)
        if not np.isfinite(term):
            out = {
                "failed": True,
                "reason": "Terme de log-vraisemblance non fini",
                "eigvals_A": lam,
                "t_fail": t,
            }
            return out if return_details else -np.inf

        loglik += term

        # Mise à jour (observation scalaire, R = 0)
        K = (P_pred @ b) / S
        x = x_pred + K * v
        P = P_pred - np.outer(K, K) * S
        P = nearest_psd(P, eps=psd_eps)

        if return_details:
            innovations[t] = v
            innovation_vars[t] = S
            x_pred_hist[t] = x_pred
            x_filt_hist[t] = x

    if return_details:
        return {
            "failed": False,
            "loglik": float(loglik),
            "innovations": innovations,
            "innovation_vars": innovation_vars,
            "x_pred": x_pred_hist,
            "x_filt": x_filt_hist,
            "A": A,
            "F": F,
            "Q": Q,
            "V_inf": V_inf,
            "eigvals_A": lam,
            "b": b,
        }

    return float(loglik)


# ============================================================
# 6) Estimation de b0 uniquement
# ============================================================

def estimate_b0_mle(
    y,
    h,
    a1,
    a2,
    a3,
    sigma2,
    bounds=(-15.0, 15.0),
    demean=True,
    return_optimizer=False,
):
    """
    Estimation MLE conditionnelle sur (a1,a2,a3,sigma2) fixés.
    """
    y = np.asarray(y, dtype=float).ravel()

    def objective(b0):
        ll = kalman_loglik_b0_only(
            y=y,
            h=h,
            a1=a1,
            a2=a2,
            a3=a3,
            sigma2=sigma2,
            b0=float(b0),
            demean=demean,
            return_details=False,
        )
        if not np.isfinite(ll):
            return 1e100
        return -ll

    res = minimize_scalar(
        objective,
        bounds=bounds,
        method="bounded",
        options={"xatol": 1e-5},
    )

    b0_hat = float(res.x)
    ll_hat = -float(res.fun) if np.isfinite(res.fun) else -np.inf

    if return_optimizer:
        return b0_hat, ll_hat, res

    return b0_hat, ll_hat


# ============================================================
# 7) Pipeline complet sur ton CSV hourly
# ============================================================

def fit_carma31_b0_from_csv(
    csv_path="../data/deseasonalised/price_resid.csv",
    value_col="price_deseasoned",
    h=1.0,
    a1=A1_FIX,
    a2=A2_FIX,
    a3=A3_FIX,
    sigma2=SIGMA2_FIX,
    bounds=(-15.0, 15.0),
    demean=True,
    return_filter_details=True,
):
    """
    Charge le CSV, suppose des données hourly (h=1),
    et estime seulement b0.
    """
    df = pd.read_csv(csv_path, index_col=0)

    if value_col not in df.columns:
        raise ValueError(f"Colonne '{value_col}' absente. Colonnes = {list(df.columns)}")

    y_series = df[value_col].astype(float).copy()
    time_index = pd.to_datetime(y_series.index, errors="coerce")

    y = y_series.to_numpy(dtype=float)

    b0_hat, ll_hat, opt = estimate_b0_mle(
        y=y,
        h=h,
        a1=a1,
        a2=a2,
        a3=a3,
        sigma2=sigma2,
        bounds=bounds,
        demean=demean,
        return_optimizer=True,
    )

    result = {
        "n_obs": len(y),
        "h": h,
        "a1": a1,
        "a2": a2,
        "a3": a3,
        "sigma2": sigma2,
        "demean": demean,
        "mean_y": float(np.mean(y)),
        "std_y": float(np.std(y, ddof=0)),
        "b0_hat": b0_hat,
        "loglik": ll_hat,
        "optimizer_success": bool(opt.success),
        "optimizer_message": opt.message,
        "time_head": time_index[:5],
    }

    if return_filter_details:
        filt = kalman_loglik_b0_only(
            y=y,
            h=h,
            a1=a1,
            a2=a2,
            a3=a3,
            sigma2=sigma2,
            b0=b0_hat,
            demean=demean,
            return_details=True,
        )
        result["filter"] = filt

    return result


# ============================================================
# 8) Exécution
# ============================================================


if __name__ == "__main__":
    res = fit_carma31_b0_from_csv(
        csv_path="../data/deseasonalised/price_resid.csv",
        value_col="price_deseasoned",
        h=1.0,  # hourly
        a1=2.94,
        a2=4.17,
        a3=0.34,
        sigma2=130.446803436953,
        bounds=(-15.0, 15.0),
        demean=True,
        return_filter_details=True,
    )

    print("========== RESULTATS ==========")
    print("Nombre d'observations :", res["n_obs"])
    print("Pas h                 :", res["h"], "hours")
    print("a1                    :", res["a1"])
    print("a2                    :", res["a2"])
    print("a3                    :", res["a3"])
    print("sigma2                :", res["sigma2"])
    print("demean                :", res["demean"])
    print("b0_hat                :", res["b0_hat"])
    print("loglik                :", res["loglik"])
    print("optim success         :", res["optimizer_success"])
    print("optim message         :", res["optimizer_message"])

    filt = res["filter"]
    print("filter failed         :", filt["failed"])

    if filt["failed"]:
        print("reason                :", filt["reason"])
        if "t_fail" in filt:
            print("t_fail                :", filt["t_fail"])
    else:
        print("5 premières innovations :", filt["innovations"][:5])
        print("5 premières variances   :", filt["innovation_vars"][:5])
        print("eig(A)                  :", filt["eigvals_A"])
        print("diag(V_inf)             :", np.diag(filt["V_inf"]))
        print("diag(Q)                 :", np.diag(filt["Q"]))

In [ ]:
import numpy as np
import pandas as pd
from scipy.linalg import expm, eigvals
from scipy.optimize import minimize


# ============================================================
# 1) Outils linéaires
# ============================================================

def matrixA(a):
    a = np.asarray(a, dtype=float)
    p = len(a)

    A = np.zeros((p, p), dtype=float)
    if p > 1:
        A[:-1, 1:] = np.eye(p - 1)
    A[-1, :] = -a[::-1]
    return A


def check_stability(A, tol=1e-12):
    lam = eigvals(A)
    lam = np.real_if_close(lam)
    stable = np.all(np.real(lam) < -tol)
    return stable, lam


def nearest_psd(M, eps=1e-12):
    M = 0.5 * (M + M.T)
    vals, vecs = np.linalg.eigh(M)
    vals = np.maximum(vals, eps)
    return (vecs * vals) @ vecs.T


# ============================================================
# 2) V_inf façon yuima
# ============================================================

def v0inf_yuima(a, sigma2):
    a = np.asarray(a, dtype=float)
    p = len(a)

    B = np.zeros((p, p), dtype=float)
    aa = -a[::-1].copy()

    for i in range(p):
        iR = i + 1
        for j in range(p):
            jR = j + 1
            k = 2 * jR - iR

            if 1 <= k <= p:
                B[i, j] = ((-1.0) ** (jR - iR)) * aa[k - 1]
            if k == p + 1:
                B[i, j] = ((-1.0) ** (jR - iR - 1))

    e_p = np.zeros(p)
    e_p[-1] = 1.0

    Vdiag = -np.linalg.solve(B, e_p) * 0.5 * sigma2
    V = np.diag(Vdiag)

    for i in range(p):
        iR = i + 1
        for j in range(i + 1, p):
            jR = j + 1
            if ((iR + jR) % 2) == 0:
                midR = (iR + jR) // 2
                sign = (-1.0) ** ((iR - jR) // 2)
                V[i, j] = sign * V[midR - 1, midR - 1]
                V[j, i] = V[i, j]

    return nearest_psd(V, eps=1e-14)


# ============================================================
# 3) Discrétisation
# ============================================================

def discrete_matrices_yuima_style(a, h, sigma2):
    A = matrixA(a)
    F = expm(A * h)
    F = np.real_if_close(F)

    V_inf = v0inf_yuima(a, sigma2)
    Q = V_inf - F @ V_inf @ F.T
    Q = nearest_psd(Q, eps=1e-14)

    return A, F, Q, V_inf


# ============================================================
# 4) Log-vraisemblance
# ============================================================

def kalman_loglik_carma31(
    y,
    h,
    a1,
    a2,
    a3,
    sigma2,
    b0,
    demean=True,
    return_details=False,
    psd_eps=1e-12,
    sn_min=1e-10,
):
    y = np.asarray(y, dtype=float).ravel()
    if y.size == 0:
        raise ValueError("y vide.")

    if demean:
        y = y - np.mean(y)

    a = np.array([a1, a2, a3], dtype=float)

    if not np.all(np.isfinite(a)) or not np.isfinite(b0) or not np.isfinite(sigma2):
        out = {"failed": True, "reason": "Paramètres non finis"}
        return out if return_details else -np.inf

    if sigma2 <= 0:
        out = {"failed": True, "reason": f"sigma2 <= 0 : {sigma2}"}
        return out if return_details else -np.inf

    try:
        A, F, Q, V_inf = discrete_matrices_yuima_style(a=a, h=h, sigma2=sigma2)
    except Exception as e:
        out = {"failed": True, "reason": f"Erreur discrétisation/V_inf: {e}"}
        return out if return_details else -np.inf

    stable, lam = check_stability(A)
    if not stable:
        out = {
            "failed": True,
            "reason": "A non stable",
            "eigvals_A": lam,
            "a": a,
        }
        return out if return_details else -np.inf

    b = np.array([b0, 1.0, 0.0], dtype=float)

    x = np.zeros(3, dtype=float)
    P = V_inf.copy()

    loglik = 0.0

    if return_details:
        innovations = np.zeros_like(y)
        innovation_vars = np.zeros_like(y)

    for t, yt in enumerate(y):
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q
        P_pred = nearest_psd(P_pred, eps=psd_eps)

        v = yt - b @ x_pred
        S = float(b @ P_pred @ b)

        if (not np.isfinite(S)) or (S <= sn_min):
            out = {
                "failed": True,
                "reason": f"Innovation variance invalide: S={S}",
                "t_fail": t,
            }
            return out if return_details else -np.inf

        term = -0.5 * (np.log(2.0 * np.pi) + np.log(S) + (v * v) / S)
        if not np.isfinite(term):
            out = {
                "failed": True,
                "reason": "Terme de log-vraisemblance non fini",
                "t_fail": t,
            }
            return out if return_details else -np.inf

        loglik += term

        K = (P_pred @ b) / S
        x = x_pred + K * v
        P = P_pred - np.outer(K, K) * S
        P = nearest_psd(P, eps=psd_eps)

        if return_details:
            innovations[t] = v
            innovation_vars[t] = S

    if return_details:
        return {
            "failed": False,
            "loglik": float(loglik),
            "innovations": innovations,
            "innovation_vars": innovation_vars,
            "eigvals_A": lam,
            "A": A,
            "F": F,
            "Q": Q,
            "V_inf": V_inf,
            "a": a,
            "b": b,
            "sigma2": sigma2,
        }

    return float(loglik)


# ============================================================
# 5) Projection dans les bornes
# ============================================================

def clip_params(theta, bounds):
    theta = np.asarray(theta, dtype=float).copy()
    for i, (lo, hi) in enumerate(bounds):
        theta[i] = np.clip(theta[i], lo, hi)
    return theta


# ============================================================
# 6) Objectif MLE
# ============================================================

def negloglik_carma31(theta, y, h, demean=True):
    a1, a2, a3, b0, sigma2 = theta

    # bornes dures dans l'objectif
    if not (-10 <= a1 <= 10 and -10 <= a2 <= 10 and -10 <= a3 <= 10 and -10 <= b0 <= 10):
        return 1e12
    if not (1e-8 <= sigma2 <= 1e4):
        return 1e12

    ll = kalman_loglik_carma31(
        y=y,
        h=h,
        a1=a1,
        a2=a2,
        a3=a3,
        sigma2=sigma2,
        b0=b0,
        demean=demean,
        return_details=False,
    )

    if not np.isfinite(ll):
        return 1e12

    return -ll


# ============================================================
# 7) Optimisation robuste multi-start
# ============================================================

def estimate_carma31_mle_full(
    y,
    h,
    init_params,
    demean=True,
    n_starts=10,
    random_state=123,
    bounds=(
        (-10.0, 10.0),  # a1
        (-10.0, 10.0),  # a2
        (-10.0, 10.0),  # a3
        (-10.0, 10.0),  # b0
        (1e-8, 1e4),    # sigma2
    ),
    return_all=False,
):
    """
    Estimation conjointe de (a1,a2,a3,b0,sigma2)
    par optimisation sans gradient + multi-start.
    """
    rng = np.random.default_rng(random_state)
    y = np.asarray(y, dtype=float).ravel()

    init_params = np.asarray(init_params, dtype=float)
    if init_params.shape != (5,):
        raise ValueError("init_params doit être de longueur 5.")

    init_params = clip_params(init_params, bounds)

    starts = [init_params]

    # quelques points de départ aléatoires autour de l'initial
    for _ in range(n_starts - 1):
        candidate = init_params.copy()
        candidate[:4] += rng.normal(0.0, 1.5, size=4)
        candidate[4] *= np.exp(rng.normal(0.0, 0.5))
        candidate = clip_params(candidate, bounds)
        starts.append(candidate)

    best_res = None
    all_res = []

    for k, x0 in enumerate(starts):
        res = minimize(
            negloglik_carma31,
            x0=x0,
            args=(y, h, demean),
            method="Powell",
            options={
                "maxiter": 3000,
                "xtol": 1e-4,
                "ftol": 1e-4,
                "disp": False,
            },
        )

        # on re-projette dans les bornes
        x_hat = clip_params(res.x, bounds)
        f_hat = negloglik_carma31(x_hat, y, h, demean)

        record = {
            "start_id": k,
            "x0": x0,
            "x_hat": x_hat,
            "fun": f_hat,
            "success": bool(res.success),
            "message": res.message,
            "nit": getattr(res, "nit", None),
            "nfev": getattr(res, "nfev", None),
        }
        all_res.append(record)

        if (best_res is None) or (f_hat < best_res["fun"]):
            best_res = record

    a1_hat, a2_hat, a3_hat, b0_hat, sigma2_hat = best_res["x_hat"]
    ll_hat = -best_res["fun"]

    out = {
        "a1_hat": float(a1_hat),
        "a2_hat": float(a2_hat),
        "a3_hat": float(a3_hat),
        "b0_hat": float(b0_hat),
        "sigma2_hat": float(sigma2_hat),
        "loglik": float(ll_hat),
        "best_start_id": best_res["start_id"],
        "best_x0": best_res["x0"],
        "optimizer_success": best_res["success"],
        "optimizer_message": best_res["message"],
        "nit": best_res["nit"],
        "nfev": best_res["nfev"],
    }

    if return_all:
        out["all_runs"] = all_res

    return out


# ============================================================
# 8) Pipeline CSV
# ============================================================

def fit_carma31_full_from_csv(
    csv_path="../data/deseasonalised/price_resid.csv",
    value_col="price_deseasoned",
    h=1.0,
    init_params=(2.94, 4.17, 0.34, 7.66, 130.446803436953),
    demean=True,
    n_starts=10,
    return_filter_details=True,
    return_all=False,
):
    df = pd.read_csv(csv_path, index_col=0)

    if value_col not in df.columns:
        raise ValueError(f"Colonne '{value_col}' absente. Colonnes = {list(df.columns)}")

    y_series = df[value_col].astype(float).copy()
    time_index = pd.to_datetime(y_series.index, errors="coerce")
    y = y_series.to_numpy(dtype=float)

    mle_res = estimate_carma31_mle_full(
        y=y,
        h=h,
        init_params=init_params,
        demean=demean,
        n_starts=n_starts,
        return_all=return_all,
    )

    result = {
        "n_obs": len(y),
        "h": h,
        "demean": demean,
        "mean_y": float(np.mean(y)),
        "std_y": float(np.std(y, ddof=0)),
        "x0_user": np.asarray(init_params, dtype=float),
        "a1_hat": mle_res["a1_hat"],
        "a2_hat": mle_res["a2_hat"],
        "a3_hat": mle_res["a3_hat"],
        "b0_hat": mle_res["b0_hat"],
        "sigma2_hat": mle_res["sigma2_hat"],
        "loglik": mle_res["loglik"],
        "best_start_id": mle_res["best_start_id"],
        "best_x0": mle_res["best_x0"],
        "optimizer_success": mle_res["optimizer_success"],
        "optimizer_message": mle_res["optimizer_message"],
        "nit": mle_res["nit"],
        "nfev": mle_res["nfev"],
        "time_head": time_index[:5],
    }

    if return_all:
        result["all_runs"] = mle_res["all_runs"]

    if return_filter_details:
        filt = kalman_loglik_carma31(
            y=y,
            h=h,
            a1=result["a1_hat"],
            a2=result["a2_hat"],
            a3=result["a3_hat"],
            sigma2=result["sigma2_hat"],
            b0=result["b0_hat"],
            demean=demean,
            return_details=True,
        )
        result["filter"] = filt

    return result


# ============================================================
# 9) Exemple
# ============================================================

if __name__ == "__main__":
    res = fit_carma31_full_from_csv(
        csv_path="../data/deseasonalised/price_resid.csv",
        value_col="price_deseasoned",
        h=1.0,
        init_params=(2.94, 4.17, 0.34, 7.66, 130.446803436953),
        demean=True,
        n_starts=20,
        return_filter_details=True,
        return_all=True,
    )

    print("========== RESULTATS ==========")
    print("x0 user                :", res["x0_user"])
    print("best x0                :", res["best_x0"])
    print("best start id          :", res["best_start_id"])
    print("a1_hat                 :", res["a1_hat"])
    print("a2_hat                 :", res["a2_hat"])
    print("a3_hat                 :", res["a3_hat"])
    print("b0_hat                 :", res["b0_hat"])
    print("sigma2_hat             :", res["sigma2_hat"])
    print("loglik                 :", res["loglik"])
    print("optim success          :", res["optimizer_success"])
    print("optim message          :", res["optimizer_message"])
    print("nit                    :", res["nit"])
    print("nfev                   :", res["nfev"])

    filt = res["filter"]
    print("filter failed          :", filt["failed"])
    if filt["failed"]:
        print("reason                 :", filt["reason"])
    else:
        print("eig(A)                 :", filt["eigvals_A"])
        print("5 innovations          :", filt["innovations"][:5])
        print("5 vars innovation      :", filt["innovation_vars"][:5])